# Movie Night Decision Agent

A simple LangChain agent that helps you pick a movie for movie night.

It has 3 tools:
1. `search_movie` — looks up a movie by title using the OMDb API
2. `get_movie_rating` — checks IMDb rating and runtime of a movie
3. `pick_random_movie` — suggests a random movie by genre (no API, local list)

The agent (powered by Groq) decides on its own which tool(s) to call based on what you ask.

## Step 1: Install packages

In [1]:
!pip install langchain langchain-groq requests -q


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 2: Set up API keys

You need two keys:
- **Groq API key** — from https://console.groq.com
- **OMDb API key** — free key from https://www.omdbapi.com/apikey.aspx

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
OMDB_API_KEY = os.getenv("OMDB_API_KEY")

## Step 3: Build the tools

Each tool is just a normal Python function with a `@tool` decorator.
The docstring tells the agent what the tool does and when to use it.

1. Search movie with OMDB API

In [2]:
import requests
from langchain.tools import tool

@tool
def search_movie(title: str) -> str:
    """Search for a movie by title and return its year, genre, and plot."""
    url = f"http://www.omdbapi.com/?t={title}&apikey={OMDB_API_KEY}"
    data = requests.get(url).json()

    if data.get("Response") == "False":
        return f"No movie found with title '{title}'"

    return (
        f"Title: {data['Title']}\n"
        f"Year: {data['Year']}\n"
        f"Genre: {data['Genre']}\n"
        f"Plot: {data['Plot']}"
    )

2. Movie rating tool

In [3]:
@tool
def get_movie_rating(title: str) -> str:
    """Get the IMDb rating and runtime of a movie by title."""
    url = f"http://www.omdbapi.com/?t={title}&apikey={OMDB_API_KEY}"
    data = requests.get(url).json() 

    if data.get("Response") == "False":
        return f"No movie found with title '{title}'"

    return f"IMDb Rating: {data['imdbRating']}, Runtime: {data['Runtime']}"

# Random movie picker tool

In [4]:
import random

MOVIE_LIST = {
    "comedy": ["Superbad", "The Hangover", "Bridesmaids"],
    "action": ["Mad Max: Fury Road", "John Wick", "Die Hard"],
    "horror": ["Hereditary", "Get Out", "A Quiet Place"],
    "romance": ["La La Land", "The Notebook", "Pride and Prejudice"],
    "sci-fi": ["Interstellar", "Arrival", "Blade Runner 2049"]
}

@tool
def pick_random_movie(genre: str) -> str:
    """Pick a random movie suggestion for a given genre, e.g. comedy, action, horror, romance, sci-fi."""
    genre = genre.lower().strip()
    if genre not in MOVIE_LIST:
        return f"No suggestions available for genre '{genre}'"
    return random.choice(MOVIE_LIST[genre])

## Step 4: Set up the LLM (Groq)

In [5]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

## Step 5: Build the agent

We give the agent the list of tools and a prompt. It will decide which tool(s) to call based on the user's question.

In [6]:
from langchain.agents import create_agent

tools = [search_movie, get_movie_rating, pick_random_movie]

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful movie night assistant. Use the tools to answer the user's question."
)

## Step 6: Try it out

In [ ]:
response = agent.invoke({"messages": [{"role": "user", "content": "Tell me about the movie Interstellar and its IMDb rating"}]})
print(response["messages"][-1].content)  

In [ ]:
response = agent.invoke({"messages": [{"role": "user", "content": "Surprise me with a comedy movie"}]})
print(response["messages"][-1].content)

In [ ]:
response = agent.invoke({"messages": [{"role": "user", "content": "Is John Wick worth watching? Check its rating and runtime."}]})
print(response["messages"][-1].content))

# Check the steps of the agent

In [ ]:
for msg in response["messages"]:
    msg.pretty_print()